# RODNet 데모 (CRUW / ROD2021)

[yizhou-wang/RODNet](https://github.com/yizhou-wang/RODNet) 공식 **추론(`tools/test.py --demo`)**을 이 프로젝트의 `ai/radar-cruw/data` 데이터와 맞춰 실행합니다.

## 사전 요구

- **NVIDIA GPU + CUDA**: 공식 `test.py`는 `torch.cuda`를 사용합니다. CPU만 있으면 추론 셀에서 오류가 납니다.
- **학습된 가중치 `.pth`**: 저장소에 고정 URL이 없을 수 있습니다. 학습 산출물 또는 배포 파일을 받아 두세요.
  - 기본 경로: `ai/radar-cruw/data/checkpoints/rodnet_cdc.pth` (또는 환경 변수 `RODNET_CHECKPOINT`)
- **인터넷**: 최초 1회 `git clone`으로 `vendor/RODNet`, `vendor/cruw-devkit`을 받습니다.

## 이 노트북이 하는 일

1. `ai/radar-cruw/vendor/` 에 RODNet·cruw-devkit 클론
2. `pip install -e` (cruw-devkit → RODNet, `setup_wo_tdc.py`를 `setup.py`로 복사)
3. `data/` 안 **TEST** 시퀀스 RAMap을 ROD2021 레이아웃(`sequences/demo/...`)으로 스테이징
4. `prepare_data.py --split demo` 로 `.pkl` 생성
5. `test.py --demo` 로 결과·시각화 이미지 생성

## 프로젝트 구조 (요약)

```
hanhwa_final/
  ai/radar-cruw/
    0_cruw_radar_training.ipynb
    1_rodnet_demo.ipynb          ← 이 파일
    requirements-cruw.txt
    data/                         ← CRUW 파일 (대용량, git 제외될 수 있음)
    vendor/                       ← 클론 (gitignore 권장)
      RODNet/
      cruw-devkit/
```

## 실행 순서

아래 셀을 **위에서부터 순서대로** 실행하세요.


## 1. 경로


In [7]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "vod-devkit").is_dir():
            return cand
        if (cand / "ai" / "radar-cruw" / "requirements-cruw.txt").is_file():
            return cand
    return p


REPO_ROOT = find_repo_root()
RADAR_CRUW = REPO_ROOT / "ai" / "radar-cruw"
DATA_DIR = Path(os.environ.get("CRUW_DATA_DIR", "").strip() or RADAR_CRUW / "data")
VENDOR_DIR = RADAR_CRUW / "vendor"
RODNET_REPO = VENDOR_DIR / "RODNet"
CRUW_DEVKIT_REPO = VENDOR_DIR / "cruw-devkit"

# ROD2021 스테이징 (base_root: sequences/, annotations/ 포함)
STAGING_ROOT = DATA_DIR / "rodnet_staging_rod2021"
PREPARED_DIR = RODNET_REPO / "data" / "prepared_demo"
RESULTS_DIR = RODNET_REPO / "results" / "notebook_demo"

RODNET_CKPT = Path(os.environ.get("RODNET_CHECKPOINT", "") or (DATA_DIR / "checkpoints" / "rodnet_cdc.pth"))
# 공식 vendor/config의 demo.seqs=[] 는 prepare_data에서 시퀀스 목록이 비게 됨 → 이 파일 사용
RODNET_CONFIG_DEMO = RADAR_CRUW / "configs" / "rodnet_cdc_win16_cruw_demo.py"

print("REPO_ROOT   :", REPO_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("VENDOR_DIR  :", VENDOR_DIR)
print("RODNET_REPO :", RODNET_REPO)
print("STAGING_ROOT:", STAGING_ROOT)
print("RODNET_CKPT :", RODNET_CKPT, "| exists:", RODNET_CKPT.is_file())
print("RODNET_CONFIG_DEMO:", RODNET_CONFIG_DEMO)


REPO_ROOT   : C:\Users\taehu\Desktop\projects\hanhwa_final
DATA_DIR    : C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data
VENDOR_DIR  : C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor
RODNET_REPO : C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet
STAGING_ROOT: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_rod2021
RODNET_CKPT : C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\checkpoints\rodnet_cdc.pth | exists: False
RODNET_CONFIG_DEMO: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\configs\rodnet_cdc_win16_cruw_demo.py


## 2. RODNet / cruw-devkit 클론 (최초 1회)

인터넷이 필요합니다. 이미 있으면 건너뜁니다.


In [8]:
VENDOR_DIR.mkdir(parents=True, exist_ok=True)


def clone_if_missing(url: str, dest: Path) -> None:
    if dest.is_dir() and (dest / ".git").is_dir():
        print("이미 있음:", dest)
        return
    subprocess.run(["git", "clone", "--depth", "1", url, str(dest)], check=True)


clone_if_missing("https://github.com/yizhou-wang/cruw-devkit.git", CRUW_DEVKIT_REPO)
clone_if_missing("https://github.com/yizhou-wang/RODNet.git", RODNET_REPO)
print("OK")


이미 있음: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\cruw-devkit
이미 있음: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet
OK


## 3. 패키지 설치

현재 Jupyter의 Python에 `cruw`, `rodnet`을 editable install 합니다. 가상환경 커널을 사용하세요.


In [9]:
import shutil as _sh

_wo = RODNET_REPO / "setup_wo_tdc.py"
_sp = RODNET_REPO / "setup.py"
if _wo.is_file():
    _sh.copyfile(_wo, _sp)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(CRUW_DEVKIT_REPO)],
    cwd=str(CRUW_DEVKIT_REPO),
    check=True,
)
req = RODNET_REPO / "requirements.txt"
if req.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(RODNET_REPO)], cwd=str(RODNET_REPO), check=True)
print("pip install OK")


pip install OK


## 4. TEST RAMap → ROD2021 스테이징

`data/**/TEST_RAD_H/**/<시퀀스>/RADAR_RA_H` 중 첫 시퀀스를 `sequences/demo/<시퀀스>/RADAR_RA_H` 로 복사합니다. (파일이 많으면 시간이 걸립니다.)


In [10]:
from typing import Optional, Tuple


def find_first_test_radar_dir(data_dir: Path) -> Optional[Tuple[Path, str]]:
    for d in sorted(data_dir.glob("**/TEST_RAD_H/**/RADAR_RA_H")):
        if d.is_dir() and d.name == "RADAR_RA_H" and any(d.glob("*.npy")):
            return d, d.parent.name
    return None


pair = find_first_test_radar_dir(DATA_DIR)
if not pair:
    raise FileNotFoundError(
        "TEST_RAD_H .../RADAR_RA_H 를 data/ 에서 찾지 못했습니다. CRUW TEST RAMap 을 받았는지 확인하세요."
    )
radar_src, seq_name = pair

if STAGING_ROOT.exists():
    shutil.rmtree(STAGING_ROOT)
STAGING_ROOT.mkdir(parents=True)
dst_radar = STAGING_ROOT / "sequences" / "demo" / seq_name / "RADAR_RA_H"
dst_radar.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(radar_src, dst_radar)
(STAGING_ROOT / "annotations" / "demo").mkdir(parents=True, exist_ok=True)

# 시각화용 카메라 캘리브 (선택). 없으면 RGB 경고만 날 수 있음.
_calib_copied = False
for cand in sorted(DATA_DIR.glob("**/CAM_CALIB")):
    calib_sub = cand / "calib"
    if calib_sub.is_dir():
        dst_calib = STAGING_ROOT / "calib"
        if dst_calib.exists():
            shutil.rmtree(dst_calib)
        shutil.copytree(calib_sub, dst_calib)
        print("calib 복사:", calib_sub, "->", dst_calib)
        _calib_copied = True
        break
if not _calib_copied:
    print("CAM_CALIB/calib 없음 — 추론은 가능, rod_viz RGB 없을 수 있음")

print("시퀀스:", seq_name)
print("RADAR 복사:", radar_src, "->", dst_radar)
print("npy 개수:", len(list(dst_radar.glob("*.npy"))))


calib 복사: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\drive-download-20260325T021904Z-1-004\CAM_CALIB\calib -> C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_rod2021\calib
시퀀스: 2019_05_28_CM1S013
RADAR 복사: C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\TEST_RAD_H-003\TEST_RAD_H\2019_05_28_CM1S013\RADAR_RA_H -> C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_rod2021\sequences\demo\2019_05_28_CM1S013\RADAR_RA_H
npy 개수: 3600


## 5. prepare_data (demo → pkl)

공식 `vendor/RODNet/configs/config_rodnet_cdc_win16.py` 의 `demo` 에 `seqs: []` 가 있으면, 스크립트가 **빈 시퀀스 목록**만 사용해 `.pkl` 이 하나도 만들어지지 않을 수 있습니다. 이 노트북은 `configs/rodnet_cdc_win16_cruw_demo.py` 로 `demo` 폴더를 자동 스캔합니다.


In [11]:
prep = RODNET_REPO / "tools" / "prepare_dataset" / "prepare_data.py"
cfg = RODNET_CONFIG_DEMO

cmd = [
    sys.executable,
    str(prep),
    "--config",
    str(cfg),
    "--data_root",
    str(STAGING_ROOT),
    "--split",
    "demo",
    "--out_data_dir",
    str(PREPARED_DIR),
    "--overwrite",
]
print("실행:", " ".join(cmd))
r = subprocess.run(cmd, cwd=str(RODNET_REPO), capture_output=True, text=True)
if r.stdout:
    print(r.stdout, end="")
if r.stderr:
    print("stderr:", r.stderr)
r.check_returncode()

demo_pkl = sorted((PREPARED_DIR / "demo").glob("*.pkl"))
print("pkl:", demo_pkl)
if not demo_pkl:
    raise RuntimeError(
        "prepare_data 후에도 .pkl이 없습니다. "
        "섹션 4 스테이징(STAGING_ROOT/sequences/demo)을 먼저 실행했는지 확인하세요."
    )


실행: c:\Users\taehu\AppData\Local\Programs\Python\Python310\python.exe C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\tools\prepare_dataset\prepare_data.py --config C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\configs\rodnet_cdc_win16_cruw_demo.py --data_root C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_rod2021 --split demo --out_data_dir C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\data\prepared_demo --overwrite
Preparing demo sets ...
Sequence C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\rodnet_staging_rod2021/sequences\demo\2019_05_28_CM1S013 saving to C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\vendor\RODNet\data\prepared_demo\demo\2019_05_28_CM1S013.pkl
pkl: [WindowsPath('C:/Users/taehu/Desktop/projects/hanhwa_final/ai/radar-cruw/vendor/RODNet/data/prepared_demo/demo/2019_05_28_CM1S013.pkl')]


## 6. 가중치 확인

`RODNET_CHECKPOINT` 또는 `data/checkpoints/rodnet_cdc.pth` 에 **config_rodnet_cdc_win16.py (CDC)** 와 맞는 `.pth` 파일이 있어야 합니다.


In [12]:
HAS_RODNET_CKPT = RODNET_CKPT.is_file()
if not HAS_RODNET_CKPT:
    print(
        "체크포인트 없음(추론 생략 가능):",
        RODNET_CKPT,
        "\n학습 산출물을 위 경로에 두거나 환경 변수 RODNET_CHECKPOINT 로 지정하세요.",
    )
else:
    print("체크포인트 OK:", RODNET_CKPT)


체크포인트 없음(추론 생략 가능): C:\Users\taehu\Desktop\projects\hanhwa_final\ai\radar-cruw\data\checkpoints\rodnet_cdc.pth 
학습 산출물을 위 경로에 두거나 환경 변수 RODNET_CHECKPOINT 로 지정하세요.


## 7. test.py --demo (GPU 필요)


In [13]:
if not RODNET_CKPT.is_file():
    print("test.py 건너뜀 — 체크포인트(.pth)가 없습니다.")
elif not list((PREPARED_DIR / "demo").glob("*.pkl")):
    print("test.py 건너뜀 — 섹션 5에서 .pkl을 먼저 생성하세요.")
else:
    test_py = RODNET_REPO / "tools" / "test.py"
    cfg = RODNET_CONFIG_DEMO

    cmd = [
        sys.executable,
        str(test_py),
        "--config",
        str(cfg),
        "--data_root",
        str(STAGING_ROOT),
        "--data_dir",
        str(PREPARED_DIR),
        "--checkpoint",
        str(RODNET_CKPT),
        "--res_dir",
        str(RESULTS_DIR),
        "--demo",
    ]
    print("실행:", " ".join(cmd))
    r = subprocess.run(cmd, cwd=str(RODNET_REPO), capture_output=True, text=True)
    if r.stdout:
        print(r.stdout, end="")
    if r.stderr:
        print("stderr:", r.stderr)
    r.check_returncode()
    print("완료. 결과:", RESULTS_DIR)


test.py 건너뜀 — 체크포인트(.pth)가 없습니다.


## 8. 결과 이미지 미리보기 (선택)


In [14]:
from IPython.display import Image, display

viz_dir = None
for p in sorted(RESULTS_DIR.rglob("rod_viz")):
    if p.is_dir():
        viz_dir = p
        break
if viz_dir:
    jpgs = sorted(viz_dir.glob("*.jpg"))
    if jpgs:
        display(Image(filename=str(jpgs[0])))
        print("표시:", jpgs[0])
    else:
        print("rod_viz 에 jpg 없음:", viz_dir)
else:
    print("rod_viz 없음 — 추론 실패 또는 경로 확인")


rod_viz 없음 — 추론 실패 또는 경로 확인
